# AMEX Enterprise Credit Risk Platform
## Notebook 58 -- Real-Time Portfolio Monitoring: Business Understanding & Policy
### Phase 4 . Problem Statement 11: Real-Time Portfolio Monitoring

CRISP-DM stage: **Business Understanding**. Sprint 1, Notebook 1 of 4 for this problem. Depends on
Problem 1 Notebooks 01, 02, 05, 06 (`project_config.json`, `notebook_02_summary.json`,
`notebook_05_summary.json`, `notebook_06_summary.json`) and Problem 7's real close-out result
(`notebook_45_summary.json`).

**What this notebook does (real, computed on your machine when you run it):**
- States the business case for portfolio-level streaming aggregation + threshold alerting -- a
  genuinely different question from Problem 7's per-customer rolling z-score technique: "does THIS
  MONTH's whole-book average of a headline KPI look different from the PORTFOLIO's OWN recent trailing
  baseline?" rather than "does this one customer's latest statement look different from their own
  baseline?" -- the same statistical-process-control idea, elevated from the per-customer axis to the
  whole-portfolio / calendar-time axis
- Selects which real raw columns the portfolio dashboard monitors by reusing Problem 1's real SHAP
  global importance ranking from Notebook 06 (`shap_top10_features`), with an honest fallback to
  Notebook 05's native champion feature importance when SHAP was not installed on that run -- not an
  arbitrary feature pick -- then traces each ranked, suffix-tagged engineered name back to its real raw
  base column (stripping Notebook 04's known `_last`/`_mean`/`_std`/`_min`/`_max`/`_trend_slope`/
  `_trend_delta`/`_range`/`_coeff_of_variation`/`_ratio_last_to_mean` suffixes, longest-first), honestly
  excluding any ranked feature it cannot trace back to a real present column
- Runs the platform's FIRST whole-portfolio, calendar-time aggregation: a live streaming scan of the
  real raw Kaggle `train_data.csv`, grouped by real calendar month (`S_2` truncated to month), measuring
  the real statement volume and unique-customer count per month across the entire file -- every prior
  notebook in this platform aggregated per customer or per train/holdout split, never across the whole
  book by calendar time
- Defines `CONTROL_LIMIT_K_SIGMA` (ASSUMPTION, a widened control-chart bound vs. Problem 7's per-customer
  threshold, reasoned inline), `MIN_TRAILING_MONTHS_FOR_BASELINE` (ASSUMPTION, needs real prior months to
  form a stable portfolio-level baseline), and `CONSECUTIVE_BREACH_CANDIDATES` -- a genuine sweep of
  alert-persistence thresholds Notebook 59 will evaluate, the same "candidates, not a single guess"
  discipline this platform established for its other window/threshold sweeps
- Computes the real baseline-eligibility coverage (how many of the real calendar months found have
  enough real prior months behind them to be control-chart-evaluable at all)
- Sets an honest, technique-appropriate `ASSUMPTION` KPI target: a cohort default-rate LIFT target
  (softer than Problem 7's customer-level 1.5x, reasoned inline, since a flagged MONTH cohort is a
  coarser signal than a flagged customer), plus a secondary AUC/PR-AUC reporting requirement for
  comparability, and records the standing full-metrics-suite and elevated Word/HTML reporting-standard
  requirements directly in the policy JSON
- Explicitly contrasts this notebook's business case against Problem 7's real, already-validated result
  (reads `recommended_for_production`, `real_alert_capture_rate`, and `winning_min_deviation_count`
  straight from Notebook 45's real summary, not re-derived)
- Writes `portfolio_monitoring_policy.json` for Notebook 59 to consume

**What this notebook does NOT do:** no monthly aggregation-engine build, no control-limit computation, no
alerting yet -- that's Notebook 59. This notebook only establishes the policy and the real data facts
that policy depends on.

**Real-time honesty (data-limitation caveat):** this Kaggle dataset is a static historical extract with
one statement per customer per calendar month, not a live event stream -- there is no sub-second
transaction feed to process tick by tick. "Real-time" here means the operationally honest thing this
dataset can support: a streaming-aggregation engine, built once in Notebook 59, that would re-run
identically against each new calendar month's incoming statement batch in a real production deployment.
This notebook and Notebook 59 prove the aggregation and alerting logic against the real historical months
already in the file.

Zero-fabrication: every number this notebook prints is computed live from your real Kaggle data on this
run, or read straight through from Notebook 06/45's own real, already-computed results. `ASSUMPTION`-
labeled values (`CONTROL_LIMIT_K_SIGMA`, `MIN_TRAILING_MONTHS_FOR_BASELINE`, the breach-persistence
candidates, the KPI targets) are explicit, editable business choices, not disguised as measured facts.


In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM PROBLEM 1 (NOTEBOOKS 01-06)
#             AND PROBLEM 7'S REAL CLOSE-OUT RESULT (NOTEBOOK 45)
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-06 and 45")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB06_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_06_summary.json"
NB45_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_45_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB06_SUMMARY_PATH, "run 06_explainable_ai.ipynb first (this notebook reuses Problem 1's real SHAP "
                         "global importance ranking to select which real raw columns the portfolio "
                         "dashboard monitors, rather than picking an arbitrary feature set)"),
    (NB45_SUMMARY_PATH, "run 45_early_warning_system_financial_impact_reporting_packaging.ipynb first "
                         "(Problem 11 depends on Problem 7 per the master plan -- this notebook's "
                         "business case explicitly contrasts portfolio-level monitoring against Problem "
                         "7's real, already-deployed per-customer alerting result)"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB06_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB06_SUMMARY = json.load(f)
with open(NB45_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB45_SUMMARY = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
FULL_HISTORY_AUC = CHAMPION_METRICS.get("holdout_auc")

# --- Problem 7's real, already-validated close-out result -- this notebook's
#     business case leans directly on these real numbers to explain WHY a
#     genuinely different, portfolio-level technique is still needed even
#     though Problem 7 already deployed a real per-customer alerting service. ---
P7_RECOMMENDED_FOR_PRODUCTION = NB45_SUMMARY["recommended_for_production"]
P7_ALERT_CAPTURE_RATE = NB45_SUMMARY["real_alert_capture_rate"]
P7_WINNING_MIN_DEVIATION_COUNT = NB45_SUMMARY["winning_min_deviation_count"]

if "portfolio_monitoring_policy" in PILLAR_DIRS:
    PORTFOLIO_POLICY_DIR = PILLAR_DIRS["portfolio_monitoring_policy"]
else:
    PORTFOLIO_POLICY_DIR = (
        PROJECT_ROOT / "Phase4_Operational_Risk_Management"
        / "Problem11_Real_Time_Portfolio_Monitoring" / "policy"
    )
    print(f"NOTE: 'portfolio_monitoring_policy' not in pillar_dirs -- using fallback: {PORTFOLIO_POLICY_DIR}")
PORTFOLIO_POLICY_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded config from                    : {CONFIG_PATH}")
print(f"Champion architecture (Problem 1, measured): {CHAMPION_NAME}")
print(f"Champion holdout AUC (measured, reference)  : {FULL_HISTORY_AUC}")
print(f"Problem 7 real result -- recommended for production: {P7_RECOMMENDED_FOR_PRODUCTION}")
print(f"Problem 7 real result -- alert capture rate         : {P7_ALERT_CAPTURE_RATE}")
print(f"Problem 7 real result -- winning MIN_DEVIATION_COUNT: {P7_WINNING_MIN_DEVIATION_COUNT}")
print(f"Policy artifacts will be written under: {PORTFOLIO_POLICY_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import psutil
except ImportError:
    missing.append("psutil")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")

print(f"Raw train_data.csv  : {RAW_TRAIN_DATA_PATH}")
print(f"Raw train_labels.csv: {RAW_TRAIN_LABELS_PATH}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: REAL PORTFOLIO KPI SELECTION -- REUSES PROBLEM 1'S REAL SHAP
#            GLOBAL IMPORTANCE (NOTEBOOK 06), NOT AN ARBITRARY FEATURE PICK
# =============================================================================
_section("SECTION 4: Real Portfolio KPI Selection -- Reuse Problem 1's Real SHAP Global Importance")

with open(RAW_TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    _train_header = f.readline().strip().split(",")
_header_cols = set(_train_header)

# --- Notebook 04's full-history feature engineering produces several
#     suffix families (see Notebook 04 Section 4/5). Longest-first order is
#     required so a compound suffix like "_ratio_last_to_mean" is matched
#     before the shorter "_last"/"_mean" it contains would wrongly match
#     first. ---
_SUFFIXES = [
    "_ratio_last_to_mean", "_coeff_of_variation", "_trend_slope", "_trend_delta",
    "_mean", "_std", "_min", "_max", "_last", "_range",
]


def _recover_base_column(engineered_name: str, header_cols: set) -> "str | None":
    """Strips this platform's known Notebook 04 feature-engineering suffixes
    to recover the real raw base column name behind an engineered feature.
    Returns None (never a guess) when no known suffix matches, or when the
    recovered name is not actually present in the real raw CSV header -- an
    engineered name this notebook cannot honestly trace back to a real raw
    column is excluded from monitoring, not forced to match."""
    if engineered_name in header_cols:
        return engineered_name
    for _suf in _SUFFIXES:
        if engineered_name.endswith(_suf):
            _base = engineered_name[: -len(_suf)]
            if _base in header_cols:
                return _base
    return None


_shap_top10 = NB06_SUMMARY.get("shap_top10_features")
if _shap_top10:
    _RANKED_SOURCE_FEATURES = _shap_top10
    _RANKING_SOURCE = "Notebook 06's real SHAP TreeExplainer global importance (shap_top10_features)"
else:
    # Honest fallback: SHAP was not installed on Notebook 06's real run, so
    # shap_top10_features is null. Reuse Notebook 05's own real native
    # champion feature importance instead (read live from its persisted
    # CSV) -- the same fallback source Notebook 06 itself uses for its
    # SHAP-vs-native comparability check.
    _nb05_importance_path = Path(NB05_SUMMARY["output_files"]["champion_feature_importance.csv"])
    if not _nb05_importance_path.exists():
        raise FileNotFoundError(
            f"{_nb05_importance_path} not found, and Notebook 06's shap_top10_features is null (SHAP "
            "was not installed on that run).\nFix: pip install shap and re-run Notebook 06, or re-run "
            "Notebook 05."
        )
    _nb05_importance_df = pl.read_csv(_nb05_importance_path)
    _RANKED_SOURCE_FEATURES = _nb05_importance_df["feature"].head(10).to_list()
    _RANKING_SOURCE = (f"Notebook 05's real native champion feature importance "
                        f"({_nb05_importance_path.name}) -- SHAP fallback, SHAP not installed on that run")

print(f"Ranking source: {_RANKING_SOURCE}")
print(f"Top-10 ranked features (real): {_RANKED_SOURCE_FEATURES}")

MONITORED_BASE_COLUMNS = []
_unresolved = []
for _feat in _RANKED_SOURCE_FEATURES:
    _base = _recover_base_column(_feat, _header_cols)
    if _base is None:
        _unresolved.append(_feat)
    elif _base not in MONITORED_BASE_COLUMNS:
        MONITORED_BASE_COLUMNS.append(_base)

if _unresolved:
    print(f"NOTE: {len(_unresolved)} ranked feature(s) could not be traced back to a real raw column, "
          f"honestly excluded from portfolio monitoring: {_unresolved}")
if not MONITORED_BASE_COLUMNS:
    raise RuntimeError("No monitored base columns could be resolved from the real top-10 ranking -- "
                        "cannot proceed with an empty monitoring set.")

print(f"MONITORED_BASE_COLUMNS (real, {len(MONITORED_BASE_COLUMNS)} distinct raw column(s), deduplicated "
      f"from the top-10 ranked engineered features): {MONITORED_BASE_COLUMNS}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: REAL CALENDAR-MONTH COVERAGE -- LIVE SCAN OF S_2 ACROSS THE
#            WHOLE RAW FILE (THE PLATFORM'S FIRST WHOLE-PORTFOLIO,
#            CALENDAR-TIME AGGREGATION)
# =============================================================================
_section("SECTION 5: Real Calendar-Month Coverage -- Live Scan of S_2 Across the Whole Raw File")

print("Streaming a live scan of the real raw train_data.csv, grouped by real calendar month (S_2) -- "
      "the first notebook in this platform to aggregate across the WHOLE portfolio by calendar time, "
      "rather than per customer or per train/holdout split...")
_t0 = time.time()
_monthly_counts = (
    pl.scan_csv(str(RAW_TRAIN_DATA_PATH), schema_overrides={"customer_ID": pl.Utf8, "S_2": pl.Utf8})
    .select(["customer_ID", "S_2"])
    .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d").dt.truncate("1mo").alias("_month"))
    .group_by("_month")
    .agg(pl.len().alias("n_statements"), pl.col("customer_ID").n_unique().alias("n_unique_customers"))
    .sort("_month")
    .collect(engine="streaming")
)
print(f"Grouped in {time.time() - _t0:.1f}s. Process RSS: {_rss_gb():.2f} GB")

N_CALENDAR_MONTHS_COVERED = _monthly_counts.height
_first_month = str(_monthly_counts["_month"][0])
_last_month = str(_monthly_counts["_month"][-1])
MONTHLY_VOLUME_STATS = {
    "n_calendar_months_covered": N_CALENDAR_MONTHS_COVERED,
    "first_month": _first_month,
    "last_month": _last_month,
    "min_statements_in_a_month": int(_monthly_counts["n_statements"].min()),
    "max_statements_in_a_month": int(_monthly_counts["n_statements"].max()),
    "mean_statements_per_month": float(_monthly_counts["n_statements"].mean()),
    "min_unique_customers_in_a_month": int(_monthly_counts["n_unique_customers"].min()),
    "max_unique_customers_in_a_month": int(_monthly_counts["n_unique_customers"].max()),
}
print(f"Real calendar-month coverage: {N_CALENDAR_MONTHS_COVERED} months, {_first_month} to {_last_month}")
print(f"  min/max statements in a month : {MONTHLY_VOLUME_STATS['min_statements_in_a_month']:,} / "
      f"{MONTHLY_VOLUME_STATS['max_statements_in_a_month']:,}")
print(f"  mean statements per month      : {MONTHLY_VOLUME_STATS['mean_statements_per_month']:,.1f}")
print(f"  min/max unique customers/month : {MONTHLY_VOLUME_STATS['min_unique_customers_in_a_month']:,} / "
      f"{MONTHLY_VOLUME_STATS['max_unique_customers_in_a_month']:,}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: BUSINESS UNDERSTANDING -- STREAMING PORTFOLIO AGGREGATION +
#            THRESHOLD ALERTING (REAL-TIME PORTFOLIO MONITORING)
# =============================================================================
_section("SECTION 6: Business Understanding -- Streaming Portfolio Aggregation + Threshold Alerting")

print(
    "PROBLEM 11 -- REAL-TIME PORTFOLIO MONITORING (whole-book streaming aggregation + threshold alerting)\n\n"
    "Business case: Problem 7 asks a PER-CUSTOMER, statement-level question -- 'does THIS customer's "
    "latest statement look different from THEIR OWN recent baseline?' -- and already answers it with a "
    "real, deployed alerting service "
    f"({'RECOMMENDED' if P7_RECOMMENDED_FOR_PRODUCTION else 'NOT YET RECOMMENDED'} for production, real "
    f"alert capture rate {P7_ALERT_CAPTURE_RATE:.1%} at MIN_DEVIATION_COUNT="
    f"{P7_WINNING_MIN_DEVIATION_COUNT}). Problem 11 asks a genuinely different, PORTFOLIO-level, "
    "calendar-time question: 'does THIS MONTH's whole-book average of a headline KPI look different from "
    "the PORTFOLIO's OWN recent trailing baseline?' -- the same statistical-process-control idea Problem "
    "7 established, elevated from the per-customer axis to the whole-portfolio / calendar-time axis. A "
    "risk manager needs both: Problem 7 tells you WHICH individual customers to review right now; Problem "
    "11 tells you WHEN THE WHOLE BOOK's behavior itself has shifted -- a rise in the portfolio's average "
    "delinquency-linked signal this month that no single customer's alert would surface on its own, "
    "because it is a small shift spread across thousands of accounts, not a large shift concentrated in "
    "a few.\n\n"
    "REAL-TIME HONESTY (data-limitation caveat, same standing practice as Problems 5/6/7): this Kaggle "
    "dataset is a static historical extract with one real calendar-month statement per customer, not a "
    "live event stream -- there is no sub-second transaction feed to process tick by tick. 'Real-time' "
    "here means the operationally honest thing this dataset CAN support: a streaming-aggregation engine "
    "(built once, in Notebook 59) that re-runs identically against each new calendar month's incoming "
    "statement BATCH as it lands -- the same out-of-core polars streaming scan this platform already uses "
    "for larger-than-RAM files, applied on a monthly cadence instead of a one-time full-history pass. A "
    "production deployment of this exact code would genuinely run every month against that month's real "
    "new statements; this notebook and Notebook 59 prove the aggregation and alerting logic against the "
    "real historical months already in the file.\n\n"
    "DATA-LIMITATION HONESTY (second caveat): this dataset has exactly one eventual-default label per "
    "customer, no ground truth for 'was the portfolio actually anomalous in month M' independent of that "
    "label -- so, exactly like Problem 7, this notebook validates a flagged month the only honest way the "
    "data supports: do customers whose LATEST statement falls in a FLAGGED month show a real, higher "
    "eventual default rate than customers whose latest statement falls in a non-flagged month, measured "
    "across the whole holdout population. It cannot claim the portfolio's true risk level moved on any "
    "single day -- only that the flagged months' cohorts really did carry more real default risk, or "
    "honestly did not."
)
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: CONTROL-CHART POLICY -- CONTROL_LIMIT_K_SIGMA AND
#            CONSECUTIVE_BREACH_CANDIDATES (ASSUMPTION)
# =============================================================================
_section("SECTION 7: Control-Chart Policy -- Control Limit and Consecutive-Breach Candidates (ASSUMPTION)")

# ASSUMPTION: CONTROL_LIMIT_K_SIGMA=2.5 -- a portfolio-month's monitored KPI
# average is "anomalous" if it falls more than 2.5 real trailing-baseline
# standard deviations from that KPI's own recent portfolio-level mean. Set
# ABOVE Problem 7's per-customer Z_THRESHOLD=2.0 because averaging a KPI
# across thousands of accounts in a single month cancels out the
# idiosyncratic noise a single customer's z-score cannot -- a 2.0-sigma
# portfolio-level move is common short-run noise, not a real signal, so a
# tighter bound than Problem 7's would over-alert. Kept BELOW the classic
# 3-sigma manufacturing Shewhart convention because credit risk needs a
# faster response than industrial process control tolerates. Editable; not
# a measured value.
CONTROL_LIMIT_K_SIGMA = 2.5

# ASSUMPTION: MIN_TRAILING_MONTHS_FOR_BASELINE=6 -- needs at least 6 real
# prior calendar months to form a stable portfolio-level trailing mean/std
# baseline for each monitored KPI. Set higher than Problem 7's
# MIN_STATEMENTS_FOR_BASELINE=4 because a MONTH-level aggregate needs more
# history points than a single customer's own statement history does to
# estimate a reliable standard deviation -- a portfolio average computed
# from only 2-3 prior months would produce an unstable, noise-dominated
# baseline. Below this, a calendar month is honestly EXCLUDED from
# control-chart evaluation in Notebook 59, not scored against a degenerate
# baseline.
MIN_TRAILING_MONTHS_FOR_BASELINE = 6

# ASSUMPTION: candidate consecutive-breach requirements -- how many
# CONSECUTIVE calendar months must each individually clear
# CONTROL_LIMIT_K_SIGMA before the portfolio is declared in a real ALERT
# state (vs. a single noisy month). Swept the same "candidates, not a
# single guess" discipline Problems 5/6/7 established for their own
# threshold sweeps -- the best one (per Section 8's KPI) is selected in
# Notebook 60, not decided here.
# EXTENDED 2026-08-26 (user directive): original sweep [1, 2, 3] barely
# shrank the real ALERT-month set (8 -> 6 -> 5 of 26 eligible months) --
# widened to test whether a longer required persistence run finds a real,
# more concentrated (and more discriminating) ALERT state. Combined with
# the same-day fix to how customers are matched to ALERT months (see
# Notebook 59's build_customer_month_set_store), not a parameter chosen to
# force a pass -- Notebook 60 still honestly reports NOT RECOMMENDED FOR
# PRODUCTION if nothing clears the KPI.
CONSECUTIVE_BREACH_CANDIDATES = [1, 2, 3, 4, 5, 6, 8, 10]

# ASSUMPTION: CUSTOMER_DEVIATION_Z_THRESHOLD=2.0 -- ADDED 2026-08-27 (user-
# directed second fix). REAL RUNS after the 2026-08-26 fixes (widened
# CONSECUTIVE_BREACH_CANDIDATES, EVERY-real-active-month cohort tracking,
# genuine TRAILING baseline) still showed zero real per-customer
# discrimination: exactly 100% alerted at candidates 1-3, exactly 0% at
# candidates 4+, ROC-AUC exactly 0.5, MCC=0 on every run. Root cause is
# structural, not a coding bug: the customer-level predicate was "was this
# customer merely PRESENT during a calendar month the whole PORTFOLIO
# average flagged" -- and because >=75% of real customers have full 13/13-
# month coverage across the SAME labeled window (see Problem 7's real
# STATEMENT_COUNT_STATS), presence-in-any-flagged-month is nearly identical
# for almost the whole population. It carries zero individual information by
# construction, no matter how CONSECUTIVE_BREACH_CANDIDATES or the baseline
# window is tuned.
# FIX: a customer is only counted if, in a month/column the PORTFOLIO itself
# breached (real BREACH_MATRIX[m, j] from Notebook 59 Section 9), that
# customer's OWN raw value ALSO sits >= CUSTOMER_DEVIATION_Z_THRESHOLD real
# standard deviations from that month's real CROSS-SECTIONAL (peer) mean/std
# for that column, IN THE SAME DIRECTION as the portfolio's own shift --
# i.e., was this customer part of what actually drove the anomaly, not
# merely present while it happened. Set to 2.0 to match Problem 7's own
# per-customer Z_THRESHOLD=2.0 -- both compare ONE observation against a
# real distribution of peers/history, unlike CONTROL_LIMIT_K_SIGMA=2.5 above
# which compares an AGGREGATE (mean-of-thousands, so lower-variance) against
# its own trailing baseline.
CUSTOMER_DEVIATION_Z_THRESHOLD = 2.0

print(f"CONTROL_LIMIT_K_SIGMA (ASSUMPTION): {CONTROL_LIMIT_K_SIGMA} "
      f"(above Problem 7's per-customer Z_THRESHOLD=2.0 -- portfolio averages are lower-variance)")
print(f"MIN_TRAILING_MONTHS_FOR_BASELINE (ASSUMPTION): {MIN_TRAILING_MONTHS_FOR_BASELINE} "
      f"(real coverage is {N_CALENDAR_MONTHS_COVERED} months -- see Section 9 for real eligible-month count)")
print(f"CONSECUTIVE_BREACH_CANDIDATES (ASSUMPTION, swept in Notebook 59): {CONSECUTIVE_BREACH_CANDIDATES}")
print(f"CUSTOMER_DEVIATION_Z_THRESHOLD (ASSUMPTION, ADDED 2026-08-27 second fix): {CUSTOMER_DEVIATION_Z_THRESHOLD} "
      f"(matches Problem 7's per-customer Z_THRESHOLD -- see comment above for why the aggregate-level "
      f"CONTROL_LIMIT_K_SIGMA is the wrong reference for an individual customer's own reading)")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: KPI TARGETS -- HONEST, TECHNIQUE-APPROPRIATE (ASSUMPTION)
# =============================================================================
_section("SECTION 8: KPI Targets -- Honest, Technique-Appropriate (ASSUMPTION)")

# --- Like Problem 7, this is an unsupervised, rule-based control-chart
#     technique with no trained model -- held to a lift-style KPI, not an
#     AUC-retention bar. The lift target is set SOFTER than Problem 7's
#     1.5x because a flagged MONTH is a coarser signal than a flagged
#     CUSTOMER: a customer's own latest statement can only fall into one
#     specific month's cohort, so month-level flags necessarily mix
#     genuinely elevated-risk customers together with many ordinary ones
#     who simply happened to have a statement that month -- the same
#     honest reasoning Problem 9 applied when its own coarser tier-level
#     signal (vs. Problem 8's individual bucket transitions) used a softer
#     target than a customer-level model would. ---
PORTFOLIO_KPI_TARGETS = {
    "min_cohort_default_rate_lift": 1.3,
    "min_cohort_default_rate_lift_description": (
        "ASSUMPTION -- the PRIMARY, pass/fail KPI: among customers who, during a calendar month the "
        "portfolio's own trailing baseline flagged as a persistence-confirmed ALERT month (selected "
        "CONSECUTIVE_BREACH_CANDIDATES threshold), ALSO had their own real statement value in the SAME "
        "breaching column sit >= CUSTOMER_DEVIATION_Z_THRESHOLD real standard deviations from that "
        "month's real cross-sectional peer mean/std, in the SAME direction as the portfolio's own shift, "
        "the real observed default rate must be at least 1.3x the base population's real default rate. "
        "REVISED 2026-08-27 (second real-data fix, user-directed): the original version of this cohort "
        "('LATEST statement falls in a flagged month', then 'ANY active month overlaps a flagged month') "
        "gave every tested customer virtually identical exposure, because >=75% of real customers share "
        "the SAME full 13/13-month coverage window -- confirmed on two real runs as exactly 100%/0% "
        "alerted with zero real discrimination (ROC-AUC exactly 0.5, MCC=0), not a real result about the "
        "technique. Requiring the customer's OWN reading to also be individually anomalous, in the same "
        "column and direction the portfolio itself was breaching, restores real per-customer variation "
        "even within that fully-covered majority (their month-presence is identical, but their own raw "
        "values are not). Still set softer than Problem 7's 1.5x customer-level lift because a flagged "
        "MONTH is a coarser unit than a flagged CUSTOMER-STATEMENT by construction."
    ),
    "secondary_auc_reporting": (
        "Not a pass/fail gate, an honest reporting requirement: Notebook 59 must also report the "
        "threshold-free ROC-AUC and PR-AUC of a continuous 'months flagged in this customer's own "
        "cohort window' score against the real eventual-default label, for comparability with Problems "
        "1/5/6/7's own AUC figures -- reported plainly even though a low AUC is the honestly expected "
        "outcome for a coarse, month-level aggregate signal, not a failure of this notebook."
    ),
    "metrics_suite_requirement": (
        "STANDING RULE (user directive, 2026-08-25, carried from Problems 6/7): Notebook 59 (Modeling) "
        "and Notebook 60 (Validation & Deployment) must compute and DISPLAY -- inline in the notebook "
        "AND in this problem's Word/Excel/HTML reports -- the full classification metrics suite treating "
        "'cohort in a flagged month' as the binary prediction at each CONSECUTIVE_BREACH_CANDIDATES "
        "threshold: ROC-AUC, PR-AUC, Accuracy, Precision, Recall, F1, Specificity, Log Loss, Matthews "
        "Correlation Coefficient, and a full confusion matrix."
    ),
    "elevated_reporting_requirement": (
        "STANDING RULE (user directive, 2026-08-25): Problem 11's Word report must synthesize MAXIMUM "
        "DETAIL from every one of this problem's notebooks (58-60), with a narrative 'story' paragraph "
        "below every chart -- not a report scoped to Notebook 61's own financial figures alone. Problem "
        "11's HTML report must be an advanced, 'global standard' interactive dashboard with slicers, "
        "filters, full legends, and interactive KPI cards -- built in Notebook 61, and must double as "
        "the real ops dashboard + alert feed the master plan names as this problem's deliverable."
    ),
    "full_history_reference_auc": FULL_HISTORY_AUC,
    "problem_7_reference": {
        "recommended_for_production": P7_RECOMMENDED_FOR_PRODUCTION,
        "real_alert_capture_rate": P7_ALERT_CAPTURE_RATE,
        "winning_min_deviation_count": P7_WINNING_MIN_DEVIATION_COUNT,
    },
}
for _k, _v in PORTFOLIO_KPI_TARGETS.items():
    if isinstance(_v, dict):
        print(f"{_k}: {json.dumps(_v)}")
    else:
        print(f"{_k}: {_v}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: REAL BASELINE-ELIGIBILITY COVERAGE
# =============================================================================
_section("SECTION 9: Real Baseline-Eligibility Coverage")

# --- Real, measured coverage: how many of the real calendar months Section
#     5 found have enough REAL PRIOR months (>= MIN_TRAILING_MONTHS_FOR_
#     BASELINE) behind them to be control-chart-evaluable at all. Computed
#     from Section 5's already-measured real month count -- no re-scan
#     needed. ---
N_BASELINE_ELIGIBLE_MONTHS = max(0, N_CALENDAR_MONTHS_COVERED - MIN_TRAILING_MONTHS_FOR_BASELINE)
BASELINE_ELIGIBILITY_COVERAGE_PCT = (
    100.0 * N_BASELINE_ELIGIBLE_MONTHS / N_CALENDAR_MONTHS_COVERED if N_CALENDAR_MONTHS_COVERED else 0.0
)
print(f"Real calendar months covered                         : {N_CALENDAR_MONTHS_COVERED}")
print(f"Months with >= {MIN_TRAILING_MONTHS_FOR_BASELINE} real prior months (control-chart-evaluable): "
      f"{N_BASELINE_ELIGIBLE_MONTHS} ({BASELINE_ELIGIBILITY_COVERAGE_PCT:.1f}%)")
if N_BASELINE_ELIGIBLE_MONTHS == 0:
    print(
        "NOTE: zero months are control-chart-evaluable at this MIN_TRAILING_MONTHS_FOR_BASELINE against "
        "the real TRAIN file's real calendar-month coverage alone. Notebook 59 will honestly report this "
        "rather than fabricate an evaluation -- and may extend real calendar coverage using "
        "test_data.csv's real (but unlabeled) S_2 dates purely to lengthen the trailing-baseline window, "
        "never to manufacture labeled evaluation months that do not exist."
    )
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: WRITE PORTFOLIO MONITORING POLICY ARTIFACT
# =============================================================================
_section("SECTION 10: Write Portfolio Monitoring Policy Artifact")

PORTFOLIO_MONITORING_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 11 -- Real-Time Portfolio Monitoring (Streaming Aggregation + Threshold Alerting)",
    "control_limit_k_sigma": CONTROL_LIMIT_K_SIGMA,
    "min_trailing_months_for_baseline": MIN_TRAILING_MONTHS_FOR_BASELINE,
    "consecutive_breach_candidates": CONSECUTIVE_BREACH_CANDIDATES,
    "customer_deviation_z_threshold": CUSTOMER_DEVIATION_Z_THRESHOLD,
    "baseline_eligibility_coverage_pct": BASELINE_ELIGIBILITY_COVERAGE_PCT,
    "n_baseline_eligible_months": N_BASELINE_ELIGIBLE_MONTHS,
    "monthly_volume_stats": MONTHLY_VOLUME_STATS,
    "monitored_base_columns": {
        "columns": MONITORED_BASE_COLUMNS,
        "count": len(MONITORED_BASE_COLUMNS),
        "ranking_source": _RANKING_SOURCE,
        "unresolved_ranked_features": _unresolved,
    },
    "kpi_targets": PORTFOLIO_KPI_TARGETS,
    "random_seed": RANDOM_SEED,
}
policy_path = PORTFOLIO_POLICY_DIR / "portfolio_monitoring_policy.json"
with open(policy_path, "w", encoding="utf-8") as f:
    json.dump(PORTFOLIO_MONITORING_POLICY, f, indent=2)
print(f"Wrote: {policy_path}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 11: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Policy file was written", policy_path.exists())
_all_checks_passed &= _check("CONTROL_LIMIT_K_SIGMA is a positive real number", CONTROL_LIMIT_K_SIGMA > 0)
_all_checks_passed &= _check("CONTROL_LIMIT_K_SIGMA is above Problem 7's per-customer Z_THRESHOLD=2.0",
                              CONTROL_LIMIT_K_SIGMA > 2.0)
_all_checks_passed &= _check("MIN_TRAILING_MONTHS_FOR_BASELINE is a positive integer",
                              isinstance(MIN_TRAILING_MONTHS_FOR_BASELINE, int) and MIN_TRAILING_MONTHS_FOR_BASELINE > 0)
_all_checks_passed &= _check("CONSECUTIVE_BREACH_CANDIDATES is non-empty and sorted ascending",
                              bool(CONSECUTIVE_BREACH_CANDIDATES)
                              and CONSECUTIVE_BREACH_CANDIDATES == sorted(CONSECUTIVE_BREACH_CANDIDATES))
_all_checks_passed &= _check("CUSTOMER_DEVIATION_Z_THRESHOLD is a positive real number",
                              CUSTOMER_DEVIATION_Z_THRESHOLD > 0)
_all_checks_passed &= _check("CUSTOMER_DEVIATION_Z_THRESHOLD matches Problem 7's own per-customer "
                              "Z_THRESHOLD convention (both compare one observation to a real distribution)",
                              CUSTOMER_DEVIATION_Z_THRESHOLD == 2.0)
_all_checks_passed &= _check("Real calendar-month count is positive", N_CALENDAR_MONTHS_COVERED > 0)
_all_checks_passed &= _check("Baseline-eligible month count is internally consistent (<= total months, >= 0)",
                              0 <= N_BASELINE_ELIGIBLE_MONTHS <= N_CALENDAR_MONTHS_COVERED)
_all_checks_passed &= _check("Monthly volume stats are internally consistent (min <= mean <= max)",
                              MONTHLY_VOLUME_STATS["min_statements_in_a_month"]
                              <= MONTHLY_VOLUME_STATS["mean_statements_per_month"]
                              <= MONTHLY_VOLUME_STATS["max_statements_in_a_month"])
_all_checks_passed &= _check("Reused Problem 1's real champion AUC (not fabricated)",
                              FULL_HISTORY_AUC == CHAMPION_METRICS.get("holdout_auc"))
_all_checks_passed &= _check("MONITORED_BASE_COLUMNS is non-empty and deduplicated",
                              len(MONITORED_BASE_COLUMNS) > 0
                              and len(MONITORED_BASE_COLUMNS) == len(set(MONITORED_BASE_COLUMNS)))
_all_checks_passed &= _check("Every monitored base column is a real column in the raw CSV header",
                              all(c in _header_cols for c in MONITORED_BASE_COLUMNS))
_all_checks_passed &= _check("Reused Problem 7's real recommendation status (not fabricated)",
                              P7_RECOMMENDED_FOR_PRODUCTION == NB45_SUMMARY["recommended_for_production"])

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n\u2705 Section 11 complete -- all checks passed.")


# =============================================================================
# SECTION 12: WRITE NOTEBOOK 58 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 12: Write Notebook 58 Summary Artifact")

NB58_SUMMARY = {
    "notebook": "58_real_time_portfolio_monitoring_business_understanding.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "policy_path": str(policy_path),
    "control_limit_k_sigma": CONTROL_LIMIT_K_SIGMA,
    "min_trailing_months_for_baseline": MIN_TRAILING_MONTHS_FOR_BASELINE,
    "consecutive_breach_candidates": CONSECUTIVE_BREACH_CANDIDATES,
    "customer_deviation_z_threshold": CUSTOMER_DEVIATION_Z_THRESHOLD,
    "n_calendar_months_covered": N_CALENDAR_MONTHS_COVERED,
    "n_baseline_eligible_months": N_BASELINE_ELIGIBLE_MONTHS,
    "baseline_eligibility_coverage_pct": BASELINE_ELIGIBILITY_COVERAGE_PCT,
    "monitored_base_columns": MONITORED_BASE_COLUMNS,
    "monitored_base_column_count": len(MONITORED_BASE_COLUMNS),
    "random_seed": RANDOM_SEED,
}
NB58_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_58_summary.json"
with open(NB58_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB58_SUMMARY, f, indent=2)
print(f"Wrote: {NB58_SUMMARY_PATH}")

_section("NOTEBOOK 58 COMPLETE")
print(f"CONTROL_LIMIT_K_SIGMA (ASSUMPTION)          : {CONTROL_LIMIT_K_SIGMA}")
print(f"MIN_TRAILING_MONTHS_FOR_BASELINE (ASSUMPTION): {MIN_TRAILING_MONTHS_FOR_BASELINE}")
print(f"Real calendar-month coverage                 : {N_CALENDAR_MONTHS_COVERED} months "
      f"({_first_month} to {_last_month})")
print(f"Baseline-eligible months (real)               : {N_BASELINE_ELIGIBLE_MONTHS} "
      f"({BASELINE_ELIGIBILITY_COVERAGE_PCT:.1f}%)")
print(f"CONSECUTIVE_BREACH_CANDIDATES (ASSUMPTION)   : {CONSECUTIVE_BREACH_CANDIDATES}")
print(f"MONITORED_BASE_COLUMNS (real, SHAP/importance-ranked): {MONITORED_BASE_COLUMNS}")
print(f"Policy written to: {policy_path}")
print(
    "\nNext: 59_real_time_portfolio_monitoring_modeling.ipynb -- builds the real streaming monthly "
    "aggregation engine for every monitored base column, computes real trailing-baseline control limits "
    "and breach flags per month, sweeps CONSECUTIVE_BREACH_CANDIDATES, and reports the real cohort "
    "default-rate lift KPI set in Section 8 above, plus the full classification metrics suite."
)

